In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, roc_curve, f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Data Exploration - Monorail

## Data Kit 01, 06, 27

In [ ]:
import pandas as pd
import os
import re

def load_Monorail(filepath):
    df = pd.read_csv(filepath)
    df = df[df['Non_Standard_Braking'] == 0]

    # Extract numeric part from filename and convert to integer
    match = re.search(r'kit(\d+)', os.path.basename(filepath))
    source = int(match.group(1)) if match else -1  # fallback to -1 if no match
    df['Source'] = source

    for col in df.select_dtypes(include='object'):
        try:
            df[col] = df[col].str.replace(' sec', '', regex=False).astype(float)
        except ValueError:
            continue
    return df


# List of file paths
filepaths = [
    'TestBrakefinal_data_kit01.csv',
    'TestBrakefinal_data_kit06.csv',
    'TestBrakefinal_data_kit27.csv'
]

# Process all files
dfs = [load_Monorail(fp) for fp in filepaths]

# Combine into one DataFrame
df_data = pd.concat(dfs, ignore_index=True)

df_data['Manual_Brake'] = df_data['LeakageLabel']
print(df_data[['Source']].value_counts())
print(df_data.shape)

### Data kit01 - Fills missing NaN with median value

In [ ]:
# from sklearn.impute import SimpleImputer
# import pandas as pd

# # Assuming df_data is already defined as in your code
# imputer = SimpleImputer(strategy='median')

# # Apply imputer to numeric columns only
# df_data[df_data.select_dtypes(include='number').columns] = imputer.fit_transform(df_data.select_dtypes(include='number'))

# # df_filtered = df_data[(df_data['WV_MeanPressure'] >= 2) & (df_data['WV_MeanPressure'] <= 3)]
# # print(df_filtered.shape)
# # df_filtered.head()

## San Donato Data

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv('model.csv')
df_raw['Malfunction'] = df_raw['Malfunction'].astype(str)

# Comparison with the sensors associated with leakages
level = [2]
df_reference = df_raw[df_raw['Sensor'].isin(level)]

# Add source column
df_reference['Source'] = '0'

# Create binary label
leakage_codes = ['H']
df_reference['Manual_Brake'] = np.where(
    df_reference['Malfunction'].isin(leakage_codes),
    'MB_Error',
    'Healthy'
)

# Aggregate delay and efficiency columns
delay_eff_map = {
    'Total_timing_delay': ['Brake_timing_delay_exp', 'Release_timing_delay_exp'],
    'Total_energy_delay': ['Brake_energy_delay_exp', 'Release_energy_delay_exp'],
    'Total_power_delay': ['Brake_power_delay_exp', 'Release_power_delay_exp'],
    'Total_power_efficiency': ['Brake_power_efficiency_exp', 'Release_power_efficiency_exp'],
    'Total_energy_efficiency': ['Brake_energy_effiency_exp', 'Release_energy_efficiency_exp']
}

for new_col, sources in delay_eff_map.items():
    df_reference[new_col] = df_reference[sources[0]] + df_reference[sources[1]]

# Drop original columns
cols_to_drop = [col for pair in delay_eff_map.values() for col in pair]
df_reference.drop(columns=cols_to_drop, inplace=True)

# Rename columns
rename_map = {
    'Release_start_pressure_delay_exp': 'Release_start_pressure_delay',
    'Buildup_end_pressure_delay_exp': 'Buildup_end_pressure_delay',
    'Weight': 'WV_MeanPressure',
    'Brake_action': 'EmergencyBrake_action'
}
df_reference.rename(columns=rename_map, inplace=True)

### Feature Engineering 

Split labeled data to features and target 

In [ ]:
common_cols = df_reference.columns.intersection(df_data.columns)

Features = df_reference[common_cols]
Features = Features.drop(columns=['Manual_Brake','WV_MeanPressure','EmergencyBrake_action'])
Parameters = df_reference[['WV_MeanPressure','EmergencyBrake_action','Brake_mode','Frequency','Sensor']]
Target = df_reference['Manual_Brake']
Target_raw = df_reference['Malfunction']

print(Features.shape)
Features.head()

Fills missing with median value instead of removing since we have little data

In [ ]:
# Impute missing (with median)
imp = SimpleImputer(strategy="median")
Features = pd.DataFrame(imp.fit_transform(Features), columns=Features.columns, index=Features.index)
#Features = Features.dropna(axis=1)

# Drop columns that are all NaN or constant
Features = Features.loc[:, Features.notna().any()]  # drop all-NaN
const_mask = Features.nunique(dropna=True) <= 1
if const_mask.any():
    Features = Features.loc[:, ~const_mask]

Features.shape
Features.columns

### Exploration - Feature Importance

In [ ]:
# === Unified Feature Selection Pipeline: RF, MI, ANOVA
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif, f_classif, RFE, RFECV
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

Xsel = Features.copy()

# Keep only numeric columns (if any non-numeric slipped in)
num_cols = [c for c in Xsel.columns if np.issubdtype(Xsel[c].dtype, np.number)]
Xsel = Xsel[num_cols].copy()

# Some selectors need scaling
sc_std = StandardScaler()
sc_rob = RobustScaler()
X_std = pd.DataFrame(sc_std.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)
X_rob = pd.DataFrame(sc_rob.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)

# if y is string, change to 0/1
y_enc = pd.Series(Target).astype("category")
if y_enc.dtype.name == "category":
    y_enc = y_enc.cat.codes  # e.g., Leakage=1, Normal=0

# For stability on tiny datasets
cv = StratifiedKFold(n_splits=min(5, max(2, np.bincount(y_enc).min())), shuffle=True, random_state=42)

# Helper to convert scores to ranks (lower rank = better)
def to_rank(series, higher_is_better=True):
    s = series.copy()
    if not higher_is_better:
        s = -s
    # rank 1 = best
    return s.rank(ascending=False, method="average")

# ---------- 1) RandomForest importance ----------
# Measures a feature's utility in improving the model's prediction accuracy (e.g., mean decrease in impurity).
# Captures feature interactions naturally; highly effective.
rf = RandomForestClassifier(n_estimators=500, random_state=42, class_weight="balanced")
rf.fit(Xsel, y_enc)
rf_imp = pd.Series(rf.feature_importances_, index=Xsel.columns, name="RF_Importance")
rf_rank = to_rank(rf_imp, higher_is_better=True).rename("RF_Rank")

# ---------- 2) Mutual Information ----------
# Measures statistical dependency (information gain) between a feature and the target.
# Captures non-linear relationships. Evaluates each feature independently; ignores feature interactions.
mi = mutual_info_classif(Xsel, y_enc, random_state=42, discrete_features=False)
mi_score = pd.Series(mi, index=Xsel.columns, name="MI_Score")
mi_rank = to_rank(mi_score, higher_is_better=True).rename("MI_Rank")

# ---------- 3) ANOVA F-test ----------
# (works best if roughly Gaussian/scaled; we used imputed data)
# Measures linear correlation between a feature and the target by comparing variance between groups to variance within groups.
# Assumes linear relationship and Gaussian distribution; ignores feature interactions.
F_vals, p_vals = f_classif(Xsel, y_enc)
f_score = pd.Series(F_vals, index=Xsel.columns, name="ANOVA_F")
f_rank = to_rank(f_score, higher_is_better=True).rename("ANOVA_Rank")

# ---------- Combine all rankings ----------
rank_table = pd.concat([rf_rank, mi_rank, f_rank,
                        rf_imp, mi_score, f_score], axis=1)

# OverallRank: average of available ranks (lower = better)
rank_cols = ["RF_Rank","MI_Rank","ANOVA_Rank"]
rank_table["OverallRank"] = rank_table[rank_cols].mean(axis=1)

# Sort and display top-N
N = 5
rank_table_sorted = rank_table.sort_values("OverallRank").head(N)
print("=== Top features by OverallRank (lower = better) ===")
display(rank_table_sorted)

plt.figure(figsize=(8, max(4, 0.35*N)))
rank_table_sorted.sort_values("OverallRank")["OverallRank"].plot(kind="barh")
plt.gca().invert_yaxis()
plt.title(f"Top {N} Features by Rank")
plt.xlabel("Rank (lower is better)")
plt.tight_layout()
plt.show()

topN_features = rank_table_sorted.index.tolist()
print("\nTopN feature list:", topN_features)

Features_reduced = Features[topN_features]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.heatmap(Features_reduced.corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
def box_target_plotter(data, target):
    for col in data.select_dtypes("number"):
        sns.boxplot(data=data, x=target, y=col)
        plt.show()

box_target_plotter(Features_reduced, Target)

In [ ]:
level = [2,3]
mal = ['0','A','B','H']
emer = [1]
mode = [0]
df = df_raw[df_raw['Sensor'].isin(level) & 
            df_raw['Malfunction'].isin(mal)]# & 
            #df_raw['Brake_action'].isin(emer) & 
            #df_raw['Brake_mode'].isin(mode)].copy() 

df['ManualLabel'] = np.where(df['Malfunction'].isin(['H']),
                             'Manual brake', 'Healthy')

Features = df.drop(columns=['Malfunction','ManualLabel','Weight','Brake_action','Brake_mode','Frequency','Sensor'])
Parameters = df[['Frequency']].copy() 
Target = df['ManualLabel']
Target_raw = df['Malfunction']

# Replace inf/-inf with NaN so imputer can handle them
Features = Features.replace([np.inf, -np.inf], np.nan)
print("Before imputation:", Features.shape)
imp = SimpleImputer(strategy="median")
Features_array = imp.fit_transform(Features)
print("After imputation:", Features_array.shape)


In [ ]:
valid_cols = Features.columns[~Features.isna().all()]
Features = pd.DataFrame(Features_array, columns=valid_cols, index=Features.index)
print("After imputation:", Features_array.shape)

In [ ]:
Features = pd.DataFrame(imp.fit_transform(Features), columns=Features.columns, index=Features.index)

Features = Features.loc[:, Features.notna().any()]
const_mask = Features.nunique(dropna=True) <= 1
if const_mask.any():
    Features = Features.loc[:, ~const_mask]

Features_reduced = Features[['First_phase_power','First_phase_half_time_ratio','First_phase_mean_curvature']].copy()

Target_name = 'ManualLabel' if Target.name is None else Target.name

Features_reduced[Target_name] = Target.values 
Features_reduced['Frequency'] = Parameters['Frequency'].values

desired_order = ['Healthy', 'Manual brake'] 

CUSTOM_HUE_PALETTE = ["#1f77b4", "#ff7f0e"]
def box_target_plotter_with_hue(data, target_col_name, hue_column, target_order=None):

    for col in data.columns.drop([target_col_name, hue_column]):
        if data[col].dtype in ['float64', 'int64']:
            plt.figure(figsize=(10, 6)) 
            
            sns.boxplot(data=data, 
                        x=target_col_name, 
                        y=col, 
                        hue=hue_column,
                        order=target_order,
                       palette=CUSTOM_HUE_PALETTE)
            
            plt.title(f'Distribution of {col} per Target, Grouped by {hue_column}')
            plt.xlabel(target_col_name)
            plt.ylabel(col)
            plt.legend(title=hue_column)
            plt.show()

box_target_plotter_with_hue(Features_reduced, Target_name, 'Frequency', target_order=desired_order)

## Combine Reference data with San Donato data

In [ ]:
# make sure the Source columns are of the same type
df_reference['Source'] = df_reference['Source'].astype(int)
df_data['Source'] = df_data['Source'].astype(int)

# Filter df_data to include only specific BC_IDs
valid_ids = ["0xf5", "0x49", "0xf9"]
df_data_subset = df_data[df_data['BC_ID'].isin(valid_ids)].copy()

# Combine common columns
common_cols = df_reference.columns.intersection(df_data.columns).tolist()

# Add DataSource column to each DataFrame
df_reference_subset = df_reference[common_cols].copy()
df_reference_subset['DataSource'] = 0

df_data_subset = df_data_subset[common_cols].copy()
df_data_subset['DataSource'] = 1

# Concatenate the two DataFrames
df_combined = pd.concat([df_reference_subset, df_data_subset], ignore_index=True)


In [ ]:
# Final label encoding
df_combined.rename(columns={'Manual_Brake': 'label'}, inplace=True)
df_combined['label'] = df_combined['label'].map({'Healthy': 0, 'MB_Error': 1})

# Convert ' sec' strings to float
for col in df_combined.select_dtypes(include='object'):
    try:
        df_combined[col] = df_combined[col].str.replace(' sec', '', regex=False).astype(float)
    except ValueError:
        continue

df_base = df_combined.copy()

Target_combined = df_base['label']
print(df_base.shape)
df_base.head()

Filter by Loaded condition of wagon (WV_MeanPressure > 1.7)

In [ ]:
df_filtered = df_combined[(df_combined['WV_MeanPressure'] >= 1.7)]
Target_data = df_combined['label']

print(df_filtered.shape)
df_filtered.head()

### Manual Braking Activation Fault Boxchart of Features 

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ---- inputs ----
# df: your combined dataframe (already built above)
# Features_reduced: list of column names you want to plot
# Target_combined = df['label']  # already defined
Features_reduced = ['First_phase_power','First_phase_half_time_ratio','First_phase_mean_curvature']

# ---- tidy data for plotting ----
plot_cols = ['label', 'DataSource'] + list(Features_reduced)
D = df_filtered[plot_cols].copy()

# Optional: restore human-readable class labels
label_map = {0: 'Healthy', 1: 'MB_Error'}
D['label'] = D['label'].map(label_map).astype('category')

# Optional: name your data sources
source_map = {0: 'Reference', 1: 'Monorail'}   # edit names if you like
D['DataSource'] = D['DataSource'].map(source_map).astype('category')

# Clean unusual numeric values
D.replace([np.inf, -np.inf], np.nan, inplace=True)

# Melt to long form: one row per (sample, feature)
D_long = D.melt(
    id_vars=['label','DataSource'],
    value_vars=Features_reduced,
    var_name='Feature', value_name='Value'
)
D_long = D_long.dropna(subset=['Value'])

g = sns.catplot(
    data=D_long, x='label', y='Value', hue='DataSource',
    col='Feature', col_wrap=3, kind='box', height=4, sharey=False
)
g.set_axis_labels("Class label", "Value")
g.set_titles("{col_name}")

Plot the Kit01 Data grouped by WV Mean Pressure

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1) Filter to DataSource == 1 ---
Data_Monorail = df_filtered.loc[df_base['DataSource'] == 1, ['label', 'Source'] + list(Features_reduced)].copy()

# Optional: readable class labels for x-axis
label_map = {0: 'Healthy', 1: 'MB_Error'}
Data_Monorail['label'] = Data_Monorail['label'].map(label_map).astype('category')

# Clean up impossible values
Data_Monorail.replace([np.inf, -np.inf], np.nan, inplace=True)

# --- 2) Melt to long form: one row per (sample, feature) ---
Data_long = Data_Monorail.melt(
    id_vars=['label', 'Source'],
    value_vars=Features_reduced,
    var_name='Feature', value_name='Value'
).dropna(subset=['Value', 'Source'])

# --- 3) Draw grouped boxplots (hue by Source) ---
sns.set_theme(style="whitegrid", palette="deep")
g = sns.catplot(
    data=Data_long, x='label', y='Value', hue='Source',
    col='Feature', col_wrap=3, kind='box', height=4, sharey=False
)
g.set_axis_labels("Class label", "Value")
g.set_titles("{col_name}")
g._legend.set_title("Source")
plt.show()

Plot Healthy only subset of Kit 01 and Reference

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ==============================================================
# 1) Prepare "Healthy only" subset
# ==============================================================
cols_needed = ['label', 'DataSource', 'WV_MeanPressure'] + list(Features_reduced)
D = df_base[cols_needed].copy()

label_map   = {0: 'Healthy', 1: 'MB_Error'}
source_map  = {0: 'Reference', 1: 'Monorail'}
D['label']       = D['label'].map(label_map).astype('category')
D['DataSource']  = D['DataSource'].map(source_map).astype('category')

# Keep only Healthy samples
D = D[D['label'] == 'Healthy']

# ==============================================================
# 2) Bin WV_MeanPressure into [<2, 2–3, >3]
# ==============================================================
edges  = [-np.inf, 2.0, 3.0, np.inf]
labels = ['< 2', '2–3', '> 3']
D['WV_bin'] = pd.cut(
    D['WV_MeanPressure'], bins=edges, labels=labels,
    right=True, include_lowest=True
)

# ==============================================================
# 3) Loop over each feature and make a separate figure
# ==============================================================
sns.set(style="whitegrid")

for feat in Features_reduced:
    plt.figure(figsize=(6, 5), dpi=300)  # increase size + resolution
    ax = sns.boxplot(
        data=D,
        x='DataSource', y=feat,
        hue='WV_bin',
        order=['Reference', 'Monorail'],
        hue_order=['< 2', '2–3', '> 3']
    )
    ax.set_title(f"{feat} — Healthy only", fontsize=13)
    ax.set_xlabel("Data Source", fontsize=11)
    ax.set_ylabel("Value", fontsize=11)
    ax.legend(title="WV Mean Pressure (bar)")
    plt.tight_layout()
    plt.show()
    # Optionally save each plot:
    # plt.savefig(f"{feat}_Healthy_boxplot.png", dpi=300, bbox_inches='tight')


### MANUAL BRAKE ACT - Variation Error of Features

In [ ]:
import numpy as np
import pandas as pd

# ---------------- safety: make sure Features_reduced is a list of strings ----------------
if isinstance(Features_reduced, (str, bytes)):
    Features_reduced = [Features_reduced]
else:
    Features_reduced = list(Features_reduced)

# Keep only columns we need (and that actually exist)
base_cols = ['label', 'DataSource', 'WV_MeanPressure']
use_cols  = [c for c in Features_reduced if c in df_filtered.columns]
missing   = sorted(set(Features_reduced) - set(use_cols))
if missing:
    print("Warning: missing features (skipped):", missing)

D = df_filtered[base_cols + use_cols].copy()

# Map labels and sources (works whether df has 0/1 or already strings)
label_map  = {0: 'Healthy', 1: 'Leakage'}
source_map = {0: 'Reference', 1: 'Monorail'}
D['label'] = D['label'].map(label_map).fillna(D['label'])
D['DataSource'] = D['DataSource'].map(source_map).fillna(D['DataSource'])

# Healthy only, WV in [2, 3]
D = D[(D['label'] == 'Healthy') &
      (D['WV_MeanPressure'] >= 2.0)].copy()

# Coerce features to numeric (avoid weird objects like "1.5 sec")
for c in use_cols:
    D[c] = pd.to_numeric(D[c], errors='coerce')
D.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop rows where ALL features are NaN (keeps rows if at least one feature is present)
D = D.dropna(subset=use_cols, how='all')

# Median per DataSource, features as rows
median_table = (
    D.groupby('DataSource')[use_cols]
      .median()
      .T
    # Ensure both columns exist; if a source is missing, you'll get NaN
    .reindex(columns=['Reference', 'Monorail'])
)

# Compute variation: |M1 - M0| / |M0| * 100
ref = median_table['Reference']
real = median_table['Monorail']

# avoid division by zero warnings
den = ref.replace(0, np.nan)
median_table['Variation_%'] = (real.sub(ref).abs().div(den.abs()).mul(100))

# Optional: sort by largest variation
median_table = median_table.sort_values('Variation_%', ascending=True)

print("=== Median comparison (Healthy; WV 2–3 bar) ===")
print(median_table.round(3))

median_table.round(3).to_csv('Median_Comparison_Healthy_WV2-3bar.csv', index=True)


### MANUAL BRAKE ACTIVATION - Boxchart

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

T_firstphase = df_data.copy()
# 10 Hz features to plot
vars_10hz = [
    'First_phase_half_time_ratio',
    'First_phase_mean_curvature',
    'First_phase_power'
]

# WV_MeanPressure bins: [<2, 2–3, >3]
bins = [-np.inf, 2, 3, np.inf]
bin_labels = ['<2', '2–3', '>3']

# --- Loop over each BC_ID present in T_firstphase ---
for bc_id in T_firstphase['BC_ID'].unique():
    df_sensor = T_firstphase[T_firstphase['BC_ID'] == bc_id].copy()
    if df_sensor.empty:
        continue

    # All rows are already Standard braking
    df_sensor['brake_type'] = 'Service'

    # WV_MeanPressure bins
    wv = pd.to_numeric(df_sensor['WV_MeanPressure'], errors='coerce')
    df_sensor['wv_bin'] = pd.cut(
        wv,
        bins=bins,
        labels=bin_labels,
        right=True,        # (a,b]
        include_lowest=True
    )

    sensor_id = str(bc_id)

    # --- Figure for this sensor: 1x3 subplots ---
    fig, axes = plt.subplots(1, len(vars_10hz), figsize=(5*len(vars_10hz), 4), sharey=False)
    axes = np.atleast_1d(axes)

    for ax, var in zip(axes, vars_10hz):
        y10 = pd.to_numeric(df_sensor[var], errors='coerce')

        df_feat = pd.DataFrame({
            'value': y10,
            'brake_type': df_sensor['brake_type'],
            'wv_bin': df_sensor['wv_bin']
        }).dropna(subset=['value', 'wv_bin'])

        if df_feat.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axis('off')
            continue

        sns.boxplot(
            data=df_feat,
            x='brake_type',      # only "Standard"
            y='value',
            hue='wv_bin',        # WV_MeanPressure bins
            ax=ax
        )

        feat_name = var.replace('_', ' ')
        ax.set_title(f'{feat_name} — Sensor {sensor_id}')
        ax.set_xlabel('Braking type')
        ax.set_ylabel('Value')
        ax.grid(True, axis='y', linestyle='--', alpha=0.4)

    # Single legend for the whole figure
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title='WV MeanPressure bin', loc='upper right')

    # Remove legends from individual subplots
    for ax in axes:
        if ax.get_legend() is not None:
            ax.get_legend().remove()

    fig.suptitle(f'First-phase (10 Hz) vs WV bins — Sensor {sensor_id}', y=1.05)
    fig.tight_layout()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

T_firstphase = df_data.copy()

# 10 Hz features to plot
vars_10hz = [
    'First_phase_half_time_ratio',
    'First_phase_mean_curvature',
    'First_phase_power'
]

# WV_MeanPressure bins: [<2, 2–3, >3]
bins = [-np.inf, 2, 3, np.inf]
bin_labels = ['<2', '2–3', '>3']

# --- Loop over each BC_ID present in T_firstphase ---
for bc_id in T_firstphase['BC_ID'].unique():
    df_sensor = T_firstphase[T_firstphase['BC_ID'] == bc_id].copy()
    if df_sensor.empty:
        continue

    # WV_MeanPressure bins
    wv = pd.to_numeric(df_sensor['WV_MeanPressure'], errors='coerce')
    df_sensor['wv_bin'] = pd.cut(
        wv,
        bins=bins,
        labels=bin_labels,
        right=True,
        include_lowest=True
    )

    sensor_id = str(bc_id)
    # Map EmergencyBrake_action to descriptive labels
    df_sensor['brake_label'] = df_sensor['EmergencyBrake_action'].map({0: 'Service', 1: 'Emergency'})
    # --- Figure for this sensor: 1x3 subplots ---
    fig, axes = plt.subplots(1, len(vars_10hz), figsize=(5*len(vars_10hz), 4), sharey=False)
    axes = np.atleast_1d(axes)

    for ax, var in zip(axes, vars_10hz):
        y10 = pd.to_numeric(df_sensor[var], errors='coerce')

        df_feat = pd.DataFrame({
            'value': y10,
            'brake_label': df_sensor['brake_label'],
            'wv_bin': df_sensor['wv_bin']
        }).dropna(subset=['value', 'wv_bin', 'brake_label'])


        if df_feat.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axis('off')
            continue

        sns.boxplot(
            data=df_feat,
            x='brake_label',
            y='value',
            hue='wv_bin',
            ax=ax
        )


        feat_name = var.replace('_', ' ')
        ax.set_title(f'{feat_name} — Sensor {sensor_id}')
        ax.set_xlabel('Brake Type')
        ax.set_ylabel('Value')
        ax.grid(True, axis='y', linestyle='--', alpha=0.4)

    # Single legend for the whole figure
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title='WV MeanPressure bin', loc='upper right')

    # Remove legends from individual subplots
    for ax in axes:
        if ax.get_legend() is not None:
            ax.get_legend().remove()

    fig.suptitle(f'First-phase (10 Hz) vs WV bins — Sensor {sensor_id}', y=1.05)
    fig.tight_layout()

In [ ]:
T_firstphase = df_data.copy()
vars_10hz = [
    'First_phase_half_time_ratio',
    'First_phase_mean_curvature',
    'First_phase_power'
]

for bc_id in T_firstphase['BC_ID'].unique():
    df_sensor = T_firstphase[T_firstphase['BC_ID'] == bc_id].copy()
    if df_sensor.empty:
        continue

    # Map EmergencyBrake_action to readable labels
    df_sensor['brake_label'] = 'service'
    sensor_id = str(bc_id)

    fig, axes = plt.subplots(1, len(vars_10hz), figsize=(5*len(vars_10hz), 4), sharey=False)
    axes = np.atleast_1d(axes)

    for ax, var in zip(axes, vars_10hz):
        y10 = pd.to_numeric(df_sensor[var], errors='coerce')

        df_feat = pd.DataFrame({
            'value': y10,
            'brake_label': df_sensor['brake_label']
        }).dropna(subset=['value', 'brake_label'])

        if df_feat.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axis('off')
            continue

        sns.boxplot(
            data=df_feat,
            x='brake_label',
            y='value',
            ax=ax
        )

        feat_name = var.replace('_', ' ')
        ax.set_title(f'{feat_name} — Sensor {sensor_id}')
        ax.set_xlabel('Brake Type')
        ax.set_ylabel('Value')
        ax.grid(True, axis='y', linestyle='--', alpha=0.4)

    fig.suptitle(f'First-phase (10 Hz) — Sensor {sensor_id}', y=1.05)
    fig.tight_layout()

# Start of Main Algorithm - Manual Braking

## Data preparation

In [ ]:
# ======================================================================
# STEP 1: LOAD AND PREPARE DATA (REFERENCE + MULTI-KIT MONORAIL)
# ======================================================================

def load_data(model_path, monorail_paths):
    """
    Load and prepare reference (model) data and Monorail data (one or more kits),
    align common columns, and return a single combined DataFrame.

    Parameters
    ----------
    model_path : str
        Path to model.csv (reference experimental campaign).
    monorail_paths : str or list of str
        Path or list of paths to Monorail TestBrakefinal_data_kitXX.csv files.

    Returns
    -------
    df_base : pandas.DataFrame
        Combined DataFrame with:
        - aligned common columns between reference and Monorail,
        - binary label (0/1) where available,
        - 'Source' column (kit ID or 0 for reference),
        - 'DataSource' column (0 = reference, 1 = Monorail).
    """

    # --------------------------------------------------------------
    # Helper: load and clean ONE Monorail file
    # --------------------------------------------------------------
    def load_Monorail(filepath: str) -> pd.DataFrame:
        df = pd.read_csv(filepath)

        # Keep only standard braking
        if 'Non_Standard_Braking' in df.columns:
            df = df[df['Non_Standard_Braking'] == 0]

        # Extract numeric kit ID from filename, e.g. "TestBrakefinal_data_kit06.csv" -> 6
        match = re.search(r'kit(\d+)', os.path.basename(filepath))
        source = int(match.group(1)) if match else -1
        df['Source'] = source

        # Convert "xx sec" string columns to float seconds where possible
        for col in df.select_dtypes(include='object'):
            try:
                df[col] = df[col].str.replace(' sec', '', regex=False).astype(float)
            except (AttributeError, ValueError):
                # AttributeError if column is not string-like; ValueError if some values cannot be cast
                continue
        df['Manual_Brake'] = df['LeakageLabel']        
        return df

    # --------------------------------------------------------------
    # 1) REFERENCE DATA: load, label, aggregate
    # --------------------------------------------------------------
    df_reference = pd.read_csv(model_path)
    df_reference['Malfunction'] = df_reference['Malfunction'].astype(str)

    # Binary label from malfunction code
    leakage_codes = ['H']
    df_reference['Manual_Brake'] = np.where(
    df_reference['Malfunction'].isin(leakage_codes),
    'MB_Error',
    'Healthy'
    )

    # Add Source = 0 for reference campaign
    df_reference['Source'] = 0

    # Aggregate delay and efficiency columns
    delay_eff_map = {
        'Total_timing_delay':      ['Brake_timing_delay_exp',      'Release_timing_delay_exp'],
        'Total_energy_delay':      ['Brake_energy_delay_exp',      'Release_energy_delay_exp'],
        'Total_power_delay':       ['Brake_power_delay_exp',       'Release_power_delay_exp'],
        'Total_power_efficiency':  ['Brake_power_efficiency_exp',  'Release_power_efficiency_exp'],
        'Total_energy_efficiency': ['Brake_energy_effiency_exp',   'Release_energy_efficiency_exp']
    }

    for new_col, (c1, c2) in delay_eff_map.items():
        # If any of these columns are missing in some version of model.csv, guard with .get
        if c1 in df_reference.columns and c2 in df_reference.columns:
            df_reference[new_col] = df_reference[c1] + df_reference[c2]

    # Drop original per-phase columns (only those that actually exist)
    cols_to_drop = [c for pair in delay_eff_map.values() for c in pair if c in df_reference.columns]
    df_reference.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    # Rename to your canonical names
    rename_map = {
        'Release_start_pressure_delay_exp': 'Release_start_pressure_delay',
        'Buildup_end_pressure_delay_exp':  'Buildup_end_pressure_delay',
        'Weight':                          'WV_MeanPressure',
        'Brake_action':                    'EmergencyBrake_action'
    }
    df_reference.rename(columns=rename_map, inplace=True)

    # --------------------------------------------------------------
    # 2) MONORAIL DATA: load one or more kit files
    # --------------------------------------------------------------
    if isinstance(monorail_paths, str):
        monorail_paths = [monorail_paths]

    dfs_mono = [load_Monorail(fp) for fp in monorail_paths]
    df_data = pd.concat(dfs_mono, ignore_index=True)

    # --------------------------------------------------------------
    # 3) ALIGN STRUCTURES AND COMBINE
    # --------------------------------------------------------------
    # Ensure 'Source' is integer in both
    df_reference['Source'] = df_reference['Source'].astype(int)
    df_data['Source']      = df_data['Source'].astype(int)
    
    # Filter df_data to include only specific BC_IDs
    valid_ids = ["0xf5", "0x49", "0xf9"]
    df_data_subset = df_data[df_data['BC_ID'].isin(valid_ids)].copy()
    
    # Columns common to BOTH datasets
    common_cols = df_reference.columns.intersection(df_data.columns).tolist()

    # Subsets with only common columns + a DataSource flag
    df_reference_subset = df_reference[common_cols].copy()
    df_reference_subset['DataSource'] = 0  # 0 = reference campaign

    df_data_subset = df_data_subset[common_cols].copy()
    df_data_subset['DataSource'] = 1       # 1 = Monorail (real-time) data

    # Stack reference + Monorail
    df_combined = pd.concat([df_reference_subset, df_data_subset], ignore_index=True)

    # Encode final label column (will be NaN for Monorail if it has no LeakageLabel)
    if 'Manual_Brake' in df_combined.columns:
        df_combined.rename(columns={'Manual_Brake': 'label'}, inplace=True)
        df_combined['label'] = df_combined['label'].map({'Healthy': 0, 'MB_Error': 1})

    # Convert any remaining "xx sec" string columns to float (esp. from model.csv)
    for col in df_combined.select_dtypes(include='object'):
        try:
            df_combined[col] = df_combined[col].str.replace(' sec', '', regex=False).astype(float)
        except (AttributeError, ValueError):
            continue
    df_combined["WV_bin"] = df_combined["WV_MeanPressure"].apply(
    lambda p: np.nan if pd.isna(p) else (0 if p < 1.6 else (2 if p > 3 else 1))
    )
    df_base = df_combined.copy()
    return df_base

model_path = 'model.csv'
monorail_paths = [
    'TestBrakefinal_data_kit01.csv',
    'TestBrakefinal_data_kit06.csv',
    'TestBrakefinal_data_kit27.csv'
]

## Data Preprocess - Train Test Split the Healthy Data 
then Inject the Faulty data (from SanDonato) to Test data

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

def preprocess_data(df, features, test_size=0.2, random_state=42):
    """
    Preprocess data for unsupervised anomaly detection.

    Returns
    -------
    X_train          : healthy-only features for training
    X_test           : features for evaluation (healthy + all unhealthy)
    y_train          : labels for X_train (all 0)
    y_test           : labels for X_test (0=healthy, 1=unhealthy)
    test_orig_index  : Series with original df indices for each row in X_test
    """

    df = df.copy()
    df["orig_index"] = df.index  # keep original row index

    # 1) Filter by WV_bin
    df_filt = df[df["WV_bin"].between(1, 2)].copy()

    # 2) Split healthy / unhealthy
    df_healthy   = df_filt[df_filt["label"] == 0]
    df_unhealthy = df_filt[df_filt["label"] == 1]

    # features
    X_healthy = df_healthy[features]
    idx_healthy = df_healthy["orig_index"]

    X_unhealthy = df_unhealthy[features]
    idx_unhealthy = df_unhealthy["orig_index"]

    # 3) Train/test split on healthy only
    X_train_h, X_test_h, idx_train_h, idx_test_h = train_test_split(
        X_healthy,
        idx_healthy,
        test_size=test_size,
        shuffle=True,
        random_state=random_state
    )

    # training set = healthy only
    X_train = X_train_h.reset_index(drop=True)
    y_train = pd.Series(0, index=X_train.index, name="label")

    # 4) test set = healthy-test + ALL unhealthy
    X_test = pd.concat([X_test_h, X_unhealthy], axis=0)
    y_test = pd.concat([
        pd.Series(0, index=X_test_h.index),
        pd.Series(1, index=X_unhealthy.index)
    ], axis=0)

    test_orig_index = pd.concat([idx_test_h, idx_unhealthy], axis=0)

    # reset indices so X_test / y_test align with position-based indexing
    X_test = X_test.reset_index(drop=True)
    y_test = y_test.reset_index(drop=True)
    test_orig_index = test_orig_index.reset_index(drop=True)

    print(f"Total samples (before WV filter): {len(df)}")
    print(f"Filtered samples (WV_bin in [1, 2]): {len(df_filt)}")
    print(f"  Healthy   : {len(df_healthy)}")
    print(f"  Unhealthy : {len(df_unhealthy)}")
    print(f"\nTRAIN: {len(X_train)} (healthy only)")
    print(f"TEST : {len(X_test)}  "
          f"(healthy={int((y_test==0).sum())}, unhealthy={int((y_test==1).sum())})")

    return X_train, X_test, y_train, y_test, test_orig_index


## Model Definition

### Defining function for Model used, Train Model, and Cross Validations

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score
)

from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor


# If you no longer need SMOTE / ImbPipeline anywhere else, you can remove these imports:
# from imblearn.over_sampling import SMOTE
# from imblearn.pipeline import Pipeline as ImbPipeline

# ============================================================================
# STEP 1: DEFINE ALL MODELS (ANOMALY ONLY)
# ============================================================================

def get_all_models(contamination=0.05):
    """
    Define all anomaly models to be tested.
    Returns dictionary of models grouped by type.
    """

    models = {
        'anomaly': {
            'Isolation Forest': IsolationForest(
                contamination=contamination,
                random_state=42,
                n_estimators=100,
                n_jobs=-1
            ),
            'One-Class SVM': OneClassSVM(
                nu=contamination,
                kernel='rbf',
                gamma='auto'
            ),
            'Local Outlier Factor': LocalOutlierFactor(
                contamination=contamination,
                novelty=True,
                n_neighbors=25
            )
        }
    }

    return models



# ============================================================================
# STEP 2: TRAIN ALL MODELS (ANOMALY ONLY)
# ============================================================================

def train_all_models(X_train_scaled, y_train, X_train_healthy_scaled,
                     contamination=0.05):
    """
    Train only anomaly models on healthy data.
    y_train is kept in the signature for compatibility, but not used for fitting.
    """

    models = get_all_models(contamination=contamination)
    trained_models = {}

    n_fault = int(np.asarray(y_train).sum())
    print(f"\nFaulty samples in training: {n_fault}")
    print("Using ANOMALY DETECTION ONLY (no supervised models).\n")

    print("=" * 60)
    print("TRAINING ANOMALY MODELS")
    print("=" * 60)

    for name, model in models['anomaly'].items():
        print(f"  - Training anomaly model: {name}")
        model.fit(X_train_healthy_scaled)
        trained_models[name] = {
            'model': model,
            'type': 'anomaly',
            'trained': True
        }

    return trained_models



# ============================================================================
# STEP 3: CROSS VALIDATION METHOD (ANOMALY MODELS)
# ============================================================================

def cross_validate_all_models_with_metrics(trained_models, X, y, k=5, out_csv=None):
    """
    Cross-validate anomaly models with multiple metrics and return
    a ranked pandas DataFrame.
    """

    X = np.asarray(X)
    y = np.asarray(y)

    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    records = []

    print("\n" + "="*80)
    print(f"RUNNING {k}-FOLD CROSS-VALIDATION FOR ANOMALY MODELS")
    print("="*80)

    for name, entry in trained_models.items():
        base_model = entry['model']
        mtype = entry['type']

        if mtype != 'anomaly':
            continue

        print(f"\n→ Evaluating {name}  (type = {mtype})")

        f1_list, prec_list, rec_list = [], [], []
        roc_list, pr_list = [], []  # reserved if you later add scores

        for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            # Train only on healthy samples in the training fold
            X_train_healthy = X_train[y_train == 0]
            model = clone(base_model)
            model.fit(X_train_healthy)

            raw_pred = model.predict(X_test)        # -1 = anomaly, 1 = normal
            y_pred = (raw_pred == -1).astype(int)   # 1 = fault, 0 = healthy

            f1_list.append(f1_score(y_test, y_pred, zero_division=0))
            prec_list.append(precision_score(y_test, y_pred, zero_division=0))
            rec_list.append(recall_score(y_test, y_pred, zero_division=0))

        record = {
            'Model': name,
            'Type': mtype,
            'F1_mean':          np.mean(f1_list),
            'F1_std':           np.std(f1_list),
            'Precision_mean':   np.mean(prec_list),
            'Recall_mean':      np.mean(rec_list),
            'ROC_AUC_mean':     np.mean(roc_list) if len(roc_list) > 0 else np.nan,
            'PR_AUC_mean':      np.mean(pr_list)  if len(pr_list) > 0 else np.nan,
        }
        records.append(record)

        print(f"  F1      per-fold: {np.round(f1_list, 3)}  | mean = {record['F1_mean']:.3f}")
        print(f"  Prec    per-fold: {np.round(prec_list, 3)} | mean = {record['Precision_mean']:.3f}")
        print(f"  Recall  per-fold: {np.round(rec_list, 3)} | mean = {record['Recall_mean']:.3f}")

    df_results = pd.DataFrame(records)
    df_results_sorted = df_results.sort_values(by='F1_mean', ascending=False).reset_index(drop=True)

    print("\n" + "="*80)
    print("CROSS-VALIDATION SUMMARY (ANOMALY MODELS, RANKED BY F1)")
    print("="*80)
    print(df_results_sorted)

    if out_csv is not None:
        df_results_sorted.to_csv(out_csv, index=False)
        print(f"\nSaved CV summary to: {out_csv}")

    return df_results_sorted


### Helper Function

In [ ]:
import numpy as np
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix
)

def evaluate_anomaly_threshold(y_true, scores, threshold, positive_label=1):
    """
    y_true : array-like, ground-truth labels (0/1, where 1 = faulty by default)
    scores : array-like, higher = more normal (like decision_function of IF/OC-SVM)
    threshold : float, classify as faulty if score < threshold
    positive_label : which label means "faulty" in y_true (default 1)
    
    Returns: dict with metrics + confusion + FPR, TPR, etc.
    """
    scores = np.asarray(scores)
    y_true = np.asarray(y_true)
    
    # predicted labels: 1 = faulty, 0 = healthy
    y_pred = (scores < threshold).astype(int)
    
    # binarize with positive_label
    y_pos = (y_true == positive_label).astype(int)
    
    # confusion matrix (TN, FP, FN, TP) w.r.t. healthy=0, faulty=1
    TN, FP, FN, TP = confusion_matrix(y_pos, y_pred).ravel()
    
    # FPR = FP / (FP + TN) on HEALTHY samples
    fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0
    tpr = TP / (TP + FN) if (TP + FN) > 0 else 0.0  # recall / sensitivity
    
    # standard metrics, with "faulty" = positive class
    prec = precision_score(y_pos, y_pred, zero_division=0)
    rec  = recall_score(y_pos, y_pred, zero_division=0)
    f1   = f1_score(y_pos, y_pred, zero_division=0)
    
    # for ROC/PR you need continuous scores; here we use them too
    try:
        roc = roc_auc_score(y_pos, -scores)  # -scores so larger = more faulty
    except ValueError:
        roc = np.nan
    try:
        ap = average_precision_score(y_pos, -scores)
    except ValueError:
        ap = np.nan
    
    return {
        "Threshold": threshold,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": roc,
        "AP": ap,
        "TN": TN, "FP": FP, "FN": FN, "TP": TP,
        "FPR": fpr,
        "TPR": tpr,
    }


### Model Hyperparameter Tuning

In [ ]:
from sklearn.ensemble import IsolationForest

def tune_isolation_forest(
    X_train_healthy,
    X_val, y_val,
    contaminations = [0.01, 0.05, 0.1, 0.2],
    n_estimators_list = [100, 200, 400],
    target_fpr = None,
    positive_label = 1
):
    """
    Grid search for IsolationForest (unsupervised), trained on healthy data,
    evaluated on labeled validation set.
    """
    results = []

    for cont in contaminations:
        for n_est in n_estimators_list:
            # ---- ensure Python int here ----
            n_est = int(n_est)

            iso = IsolationForest(
                contamination=cont,
                n_estimators=n_est,
                random_state=42,
                n_jobs=-1
            )
            iso.fit(X_train_healthy)
            
            scores_val = iso.decision_function(X_val)  # higher = more normal
            thresholds = np.percentile(scores_val, [1, 5, 10, 20, 30, 40, 50])
            thresholds = np.unique(np.r_[thresholds, 0.0])  # include 0 as reference

            for thr in thresholds:
                res = evaluate_anomaly_threshold(
                    y_true=y_val,
                    scores=scores_val,
                    threshold=thr,
                    positive_label=positive_label
                )
                res.update({
                    "contamination": cont,
                    "n_estimators": n_est,
                })
                results.append(res)

    df = pd.DataFrame(results)

    # pick best according to F1, with optional FPR constraint
    if target_fpr is not None:
        feasible = df[df["FPR"] <= target_fpr]
        if len(feasible) == 0:
            best_row = df.sort_values("F1", ascending=False).iloc[0]
        else:
            best_row = feasible.sort_values("F1", ascending=False).iloc[0]
    else:
        best_row = df.sort_values("F1", ascending=False).iloc[0]

    best_cont = best_row["contamination"]
    # ---- cast from pandas float to int ----
    best_n_est = int(best_row["n_estimators"])
    best_thr = best_row["Threshold"]

    best_iso = IsolationForest(
        contamination=best_cont,
        n_estimators=best_n_est,
        random_state=42,
        n_jobs=-1
    )
    best_iso.fit(X_train_healthy)

    return best_iso, best_thr, df, best_row


In [ ]:
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np

def tune_ocsvm(
    X_train_healthy,
    X_val, y_val,
    nus = [0.01, 0.05, 0.1, 0.2],
    gammas = ['scale', 'auto', 0.01, 0.1, 1.0],
    target_fpr = None,
    positive_label = 1
):
    """
    Grid search for OC-SVM using healthy-only training set + labeled validation set.
    If target_fpr is given (e.g. 0.01), we pick among candidates that respect it
    (FPR <= target_fpr) the one with best F1; otherwise best F1 overall.
    """
    results = []

    for nu in nus:
        for gamma in gammas:
            # pipeline: scale + OC-SVM
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("ocsvm", OneClassSVM(kernel='rbf', nu=nu, gamma=gamma))
            ])

            # fit on healthy-only data
            pipe.fit(X_train_healthy)

            # anomaly score (higher = more normal)
            scores_val = pipe.decision_function(X_val)

            # You can pick threshold = 0 (natural decision boundary)
            # or search a few thresholds around 0, depending on behaviour
            thresholds = np.percentile(scores_val, [1, 5, 10, 20, 30, 40, 50])
            thresholds = np.unique(np.r_[thresholds, 0.0])  # ensure 0 is included

            for thr in thresholds:
                res = evaluate_anomaly_threshold(
                    y_true=y_val,
                    scores=scores_val,
                    threshold=thr,
                    positive_label=positive_label
                )
                res.update({
                    "nu": nu,
                    "gamma": gamma,
                })
                results.append(res)

    df = pd.DataFrame(results)

    # choose best model
    if target_fpr is not None:
        # filter by FPR constraint
        feasible = df[df["FPR"] <= target_fpr]
        if len(feasible) == 0:
            # fallback: no model meets FPR constraint, pick best F1 overall
            best_row = df.sort_values("F1", ascending=False).iloc[0]
        else:
            best_row = feasible.sort_values("F1", ascending=False).iloc[0]
    else:
        best_row = df.sort_values("F1", ascending=False).iloc[0]

    best_nu = best_row["nu"]
    best_gamma = best_row["gamma"]
    best_thr = best_row["Threshold"]

    # refit best model on healthy train
    best_model = Pipeline([
        ("scaler", StandardScaler()),
        ("ocsvm", OneClassSVM(kernel='rbf', nu=best_nu, gamma=best_gamma))
    ])
    best_model.fit(X_train_healthy)

    return best_model, best_thr, df, best_row


## Algorithm 1 - Use only First Phase Mean Curvature

In [ ]:
df = load_data(model_path, monorail_paths)

print(df.shape)
print(df['DataSource'].value_counts(dropna=False))
print(df['label'].value_counts(dropna=False))  # will include NaN for unlabeled Monorail
df["orig_index"] = df.index 

In [ ]:
# ============================================================================
# STEP 1: DATA PREPROCESSING AND FEATURE SELECTION
# ============================================================================
features = [
    "First_phase_mean_curvature",
    "First_phase_power",
    "First_phase_half_time_ratio"
]

X_train, X_test, y_train, y_test, test_orig_index = preprocess_data(
    df, features, test_size=0.2, random_state=42
)

In [ ]:
import numpy as np

# positions of unhealthy samples *inside TEST*
fault_idx = np.where(y_test == 1)[0]
print("Faulty samples in TEST positions:", fault_idx)

# original row indices in the full df
orig_fault_idx = test_orig_index.iloc[fault_idx].values
print("Original dataframe indices:", orig_fault_idx)

# inspect those rows in the original dataframe
df.loc[orig_fault_idx]


In [ ]:
from sklearn.ensemble import IsolationForest
import numpy as np

iso = IsolationForest(contamination=0.15, random_state=42)
iso.fit(X_train)   # healthy-only training

scores = iso.decision_function(X_test)    # higher = more normal
raw_pred = iso.predict(X_test)            # 1 = inlier, -1 = outlier

# MAPPING
# outlier  (-1) → 1 (faulty)
# inlier    (1) → 0 (healthy)
y_pred = np.where(raw_pred == -1, 1, 0)

# find faulty samples in TEST
fault_idx = np.where(y_test == 1)[0]  # now 1 = faulty
print("Faulty sample indices:", fault_idx)
print("Faulty scores:", scores[fault_idx])
print("Faulty predictions:", y_pred[fault_idx])


In [ ]:
order = np.argsort(scores)  # lowest = most anomalous
print(order[:10])           # 10 most anomalous


In [ ]:
healthy_scores = scores[y_test == 0]  # only healthy
threshold = np.percentile(healthy_scores, 5)  # 1% FP rate
print("Threshold:", threshold)


In [ ]:
# original row indices in the full df
orig_fault_idx = test_orig_index.iloc[fault_idx].values
print("Original dataframe indices:", orig_fault_idx)

# inspect those rows in the original dataframe
df.loc[orig_fault_idx]


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred, labels=[0,1])
print(cm)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Healthy (Pred)', 'Faulty (Pred)'],
            yticklabels=['Healthy (True)', 'Faulty (True)'])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Isolation Forest Confusion Matrix (1 = Faulty)")
plt.show()



In [ ]:
# ============================================================================
# STEP 2: IMPUTE + FEATURE SCALING
# ============================================================================

from sklearn.impute import SimpleImputer

def scale_features(X_train, X_test, X_train_healthy):
    """
    Impute missing values by median, then standardize features.
    Returns imputed+scaled arrays, plus fitted scaler and imputer.
    """
    # 1) Median imputation (fit only on training set)
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)
    X_train_healthy_imp = imputer.transform(X_train_healthy)

    # 2) Standardization (fit only on imputed training set)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled = scaler.transform(X_test_imp)
    X_train_healthy_scaled = scaler.transform(X_train_healthy_imp)

    return X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer

[X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer] = scale_features(X_train, X_test, X_train)

### Use simple Percentile Based threshold on FP_Mean_Curvature

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

def percentile_threshold(x, pct=95):
    """
    Compute anomaly threshold based on a percentile cut.
    All samples above this percentile are flagged as anomalies.
    """
    x = np.asarray(x).ravel()       # ensure 1D
    thr = np.percentile(x, pct)
    return thr

def detect_anomaly(x, threshold):
    """
    Returns a binary anomaly vector (1 = anomaly, 0 = normal)
    """
    x = np.asarray(x).ravel()       # ensure 1D
    return (x < threshold).astype(int)

# ======================
# Usage
# ======================

# 1) Impute training + test
imputer = SimpleImputer(strategy='median')
X_train_simple = imputer.fit_transform(X_train)
X_test_simple  = imputer.transform(X_test)

# 2) Keep ONLY feature 0 (First_phase_mean_curvature)
X_train_feat0 = X_train_simple[:, 0]
X_test_feat0  = X_test_simple[:, 0]

# 3) Threshold from TRAIN (feature 0)
threshold = percentile_threshold(X_train_feat0, pct=14)

# 4) Detect anomalies on TEST (same feature)
anomaly_flag = detect_anomaly(X_test_feat0, threshold)

print("Selected threshold:", threshold)
print("Anomalies detected:", int(anomaly_flag.sum()))


In [ ]:
# After computing:
# X_test_feat0, anomaly_flag, threshold

# Convert anomaly_flag → Series with correct indexing
anomaly_series = pd.Series(anomaly_flag, index=X_test.index, name="is_anomaly")

# Extract rows that are anomalous
anomalies = X_test.loc[anomaly_series == 1]

print("Threshold:", threshold)
print("Number of anomalies:", len(anomalies))
print("\nAnomalous rows with original indices:")
print(anomalies)


### Train the Model

In [ ]:
# Use X_train_scaled for One Class SVM, then use X_test, y_test to validate directly
best_ocsvm, best_thr_ocsvm, ocsvm_results, ocsvm_best_row = tune_ocsvm(
    X_train_scaled,
    X_val=X_test_scaled,       # validation set
    y_val=y_test,
    nus=[0.01, 0.05, 0.1, 0.2],
    gammas=['scale', 'auto', 0.01, 0.1],
    target_fpr=0.01,   # example set max 1% false positives
    positive_label=1   # label that means "faulty"
)

print("Best OC-SVM params:")
print("  nu      =", ocsvm_best_row["nu"])
print("  gamma   =", ocsvm_best_row["gamma"])
print("  thr     =", ocsvm_best_row["Threshold"])
print("  F1      =", ocsvm_best_row["F1"])
print("  FPR     =", ocsvm_best_row["FPR"])
print("  Recall  =", ocsvm_best_row["Recall"])


In [ ]:
scores_test_oc = best_ocsvm.decision_function(X_test_scaled)
eval_oc_test = evaluate_anomaly_threshold(
    y_true=y_test,
    scores=scores_test_oc,
    threshold=best_thr_ocsvm,
    positive_label=1
)
eval_oc_test


In [ ]:
best_iso, best_thr_iso, iso_results, iso_best_row = tune_isolation_forest(
    X_train,
    X_val=X_test,
    y_val=y_test,
    contaminations=[0.01, 0.05, 0.1, 0.2],
    n_estimators_list=[100, 200, 400],
    target_fpr=None,
    positive_label=1
)

print("Best IF params:", iso_best_row)


In [ ]:

scores_test_iso = best_iso.decision_function(X_test)
eval_iso_test = evaluate_anomaly_threshold(
    y_true=y_test,
    scores=scores_test_iso,
    threshold=best_thr_iso,
    positive_label=1
)
eval_iso_test


### Plot Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ["OC-SVM", "Isolation Forest"]

f1_scores = [eval_oc_test["F1"], eval_iso_test["F1"]]
fprs      = [eval_oc_test["FPR"], eval_iso_test["FPR"]]
recalls   = [eval_oc_test["Recall"], eval_iso_test["Recall"]]

x = np.arange(len(models))
width = 0.25

plt.figure(figsize=(8, 4))
plt.bar(x - width, f1_scores, width, label='F1')
plt.bar(x,         recalls,  width, label='Recall')
plt.bar(x + width, fprs,     width, label='FPR')
plt.xticks(x, models)
plt.ylabel("Score")
plt.title("OC-SVM vs IsolationForest (Test set)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

y_pos = (y_test == 1).astype(int)

# OC-SVM: remember, larger = more normal, so flip sign
fpr_oc, tpr_oc, _ = roc_curve(y_pos, -scores_test_oc)
prec_oc, rec_oc, _ = precision_recall_curve(y_pos, -scores_test_oc)

fpr_iso, tpr_iso, _ = roc_curve(y_pos, -scores_test_iso)
prec_iso, rec_iso, _ = precision_recall_curve(y_pos, -scores_test_iso)

plt.figure(figsize=(6,5))
plt.plot(fpr_oc, tpr_oc, label=f"OC-SVM (AUC={eval_oc_test['ROC_AUC']:.3f})")
plt.plot(fpr_iso, tpr_iso, label=f"IF (AUC={eval_iso_test['ROC_AUC']:.3f})")
plt.plot([0,1],[0,1],'--',linewidth=0.8)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves (fault = positive)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,5))
plt.plot(rec_oc, prec_oc, label=f"OC-SVM (AP={eval_oc_test['AP']:.3f})")
plt.plot(rec_iso, prec_iso, label=f"IF (AP={eval_iso_test['AP']:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall curves (fault = positive)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
